# Pull deps

In [ ]:
import sys
!{sys.executable} -m pip install torchaudio torchcodec datasets huggingface_hub scipy librosa "numpy<2.3"


# Setup HF's storage manually (I need a separated HDD)

In [ ]:
from pathlib import Path

import os


In [ ]:
CACHE_DIR = Path("./.cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# Set root cache for all HF libs (datasets, hub, transformers, etc.)
_p = str(CACHE_DIR.resolve())
os.environ["HF_HOME"] = _p

print("Set HF home dir to", _p)


# Setting up

In [ ]:
from subprocess import Popen, PIPE, CalledProcessError, run
from huggingface_hub import HfApi, list_repo_files
from datasets import Dataset, Audio, ClassLabel, load_dataset, concatenate_datasets, Value

from datasets.dataset_dict import DatasetDict
from pathlib import Path
from IPython.display import Audio as IPA, display
from concurrent.futures import ThreadPoolExecutor, as_completed
import datasets

import gc
import os
import sys
import glob
import librosa
import datasets
import numpy as np
import scipy.io as sio
import pandas as pd


In [ ]:
REPO_ID = "Hibou-Foundation/datian"
LOCAL_DIR = Path("./prepared_dataset")
REPO_TYPE = "dataset"


In [ ]:
#The following DSs rely on these values
#geronimobasso/drone-audio-detection-samples
#yehiellevi/dataset-balanced-n-weighted-final
#Mixed datasets also follow this convention (at least the ones we currently have)
CLASSES_INV = {0:"other", 1: "drone"}
CLASSES = {"other": 0, "drone": 1}


In [ ]:
dl_dir = Path("./downloaded_datasets")
local_tmp_dir = Path("./staging")


In [ ]:
dl_dir.mkdir(parents=True, exist_ok=True)
LOCAL_DIR.mkdir(parents=True, exist_ok=True)
local_tmp_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
def easy_display_audio(audio, sample_rate):
    if np.max(np.abs(audio)) > 1:
        audio = audio / np.max(np.abs(audio))

    # Display the audio player
    display(IPA(data=audio, rate=int(sample_rate.item())))


# Sources

In [ ]:
download_args = [
    ["--insecure", "https://lambda-iot.uniud.it/UAV_DeepAcousticLocalizationRecognition_Datasets/Scenario%202%20Dataset.zip", "--output", f"{dl_dir}/scenario2.zip"],
    ["--insecure", "https://lambda-iot.uniud.it/UAV_DeepAcousticLocalizationRecognition_Datasets/Scenario%201%20Dataset.zip", "--output", f"{dl_dir}/scenario1.zip"],
    ["https://github.com/DroneDetectionThesis/Drone-detection-dataset/archive/refs/heads/master.zip", "--output", f"{dl_dir}/DroneDetectionThesis.Drone-detection-dataset.zip"],
    ["https://uc90e6a3b40e150d234227daf272.dl.dropboxusercontent.com/cd/0/get/C7eK5DuGCpRv4uV9pfWTsFUjgjNtBXCZpqBEFPVxcwY-jwBmfseQgQotPOZ3iBQQfdKJXZSjhkXlQVL5PclGCNhR7Sd3uxEBg9IeuD5RoBWJY7wH92mlASk-I-Y8Xsyynx18jNUG7gU7QP7XI5THb_zm/file?_download_id=1637074187773826591426354030248830796317808430536170451142051367&_log_download_success=1&_notify_domain=www.dropbox.com&dl=1", "--output", f"{dl_dir}/aira-uas.tar.gz"],
    ["https://zenodo.org/records/15391924/files/Microphone_array.zip?download=1", "--output", f"{dl_dir}/UaVirBASE.zip"],
    ["https://www.kaggle.com/api/v1/datasets/download/yehiellevi/dataset-balanced-n-weighted-final", "--output", f"{dl_dir}/yehiellevi.dataset-balanced-n-weighted-final.zip"],
    ["https://data.nasa.gov/docs/datasets/rfk401li/small_uav_acoustics.zip", "--output", f"{dl_dir}/nasa.small_uav_flyover_acoustics.zip"],
]

file_names = [
    "scenario2.zip",
    "scenario1.zip",
    "UaVirBASE.zip",
    "DroneDetectionThesis.Drone-detection-dataset.zip",
    "aira-uas.tar.gz",
    "nasa.small_uav_flyover_acoustics.zip",
    "yehiellevi.dataset-balanced-n-weighted-final.zip",
]


In [ ]:
hf_mixed_sources = [
    "geronimobasso/drone-audio-detection-samples",
]

# ("ahlab-drone-project/DroneAudioSet", ['drone-only', 'drone-with-source', 'ground-truth', 'source-only'])
# drone-with-source has been selected because of its noisy mixtures.

hf_drone_sources = [
    ("ahlab-drone-project/DroneAudioSet", ["drone-with-source"]),
]

#    "sps44/fsdnoisy18k",
hf_other_sources = [
    "agkphysics/AudioSet",
    "FluidInference/musan",
]


# Download DS archives

In [ ]:
last_dl = 0


In [ ]:
def run_command(cmd, dry=False):
    if dry:
        print(" ".join(cmd))
        return None

    with Popen(cmd, stdout=PIPE, bufsize=1, universal_newlines=True) as p:
        for line in p.stdout:
            print(line, end='') # process line here

        if p.returncode != 0 and p.returncode is not None:
            raise CalledProcessError(p.returncode, p.args)

        if p.returncode is not None:
            print("Exited with code", p.returncode)


In [ ]:
def unpack_archives(files, dry=True):
    for f in files:
        print("Unpacking", f)
        if f.endswith(".zip"):
            run_command(["unzip", str(dl_dir / f), "-d", f"{dl_dir}/{f[:-4]}"], dry)
        elif f.endswith(".tar.gz"):
            run_command(["mkdir", "-p", f"{dl_dir}/{f[:-7]}"], dry)
            run_command(["tar", "-zxvf", str(dl_dir / f), "-C", f"{dl_dir}/{f[:-7]}", "--strip-components=1"], dry)


In [ ]:
for i in range(last_dl, len(download_args)):
    run_command(["curl"] + download_args[i])
    last_dl = i


In [ ]:
unpack_archives(file_names[2:], dry=False)


# Pull in the HF datasets

In [ ]:
hf_other = [load_dataset(ds) for ds in hf_other_sources]


In [ ]:
hf_mixed = [load_dataset(ds) for ds in hf_mixed_sources]


In [ ]:
hf_drone = [load_dataset(ds) if ds is str else [load_dataset(ds[0], conf) for conf in ds[1]] for ds in hf_drone_sources]


# Process the downloaded archives

In [ ]:
def gen_aira_uas():
    data_root = f"{dl_dir}/aira-uas/"

    protos = [f"Protocol{i}" for i in range(1, 4)]
    infos = {"Protocol1": ([i for i in range(4, 16)], 18), "Protocol2": ([18], 3), "Protocol3": ([21], 3)}
    
    drones = []
    others = []
    elements = []
    
    j = 0
    for i in range(1, 4):
        subp = f"Protocol{i}"
        for k in range(infos[subp][1]):
            path = data_root + subp + f"/Recording{j}"
            if j in infos[subp][0]:
                drones += glob.glob(path + "/*.wav")
            else:
                others += glob.glob(path + "/*.wav")
            j += 1

    for f, label in [(f, CLASSES["drone"]) for f in drones] + [(f, CLASSES["other"]) for f in others]:
        data, sr = librosa.load(f, sr=None, mono=False)
        cast = np.array(data, dtype=np.float32)
        if cast.ndim > 1:
            for i in range(cast.shape[0]):
                elements.append({"audio": {"array": cast[i], "sampling_rate": sr}, "label": label, "src": "aira"})
        else:
            elements.append({"audio": {"array": cast, "sampling_rate": sr}, "label": label, "src": "aira"})

    return Dataset.from_list(elements).cast_column("audio", Audio(decode=True))


In [ ]:
def gen_ddd():
    data_root = f"{dl_dir}/DroneDetectionThesis.Drone-detection-dataset/Drone-detection-dataset-master/Data/Audio/"

    drones = glob.glob(data_root + "DRONE_*.wav")
    others = glob.glob(data_root + "BACKGROUND_*.wav") + glob.glob(data_root + "HELICOPTER_*.wav") + glob.glob(data_root + "DRONE_*.wav")

    elements = []
    
    for f in drones:
        data, sr = librosa.load(f, sr=None, mono=False)
        cast = np.array(data, dtype=np.float32)
        if cast.ndim > 1:
            for i in range(cast.shape[0]):
                elements.append({"audio": {"array": cast[i], "sampling_rate": sr}, "label": CLASSES["drone"]})
        else:
            elements.append({"audio": {"array": cast, "sampling_rate": sr}, "label": CLASSES["drone"]})

    for f in others:
        data, sr = librosa.load(f, sr=None, mono=False)
        cast = np.array(data, dtype=np.float32)
        if cast.ndim > 1:
            for i in range(cast.shape[0]):
                elements.append({"audio": {"array": cast[i], "sampling_rate": sr}, "label": CLASSES["other"], "src": "ddd"})
        else:
            elements.append({"audio": {"array": cast, "sampling_rate": sr}, "label": CLASSES["other"], "src": "ddd"})

    return Dataset.from_list(elements).cast_column("audio", Audio(decode=True))


In [ ]:
def gen_dalrd():
    data_root_1 = f"{dl_dir}/scenario1/Scenario 1 Dataset/"
    data_root_2 = f"{dl_dir}/scenario2/Scenario 2 Dataset/"
    
    drones = glob.glob(data_root_1 + "*.wav", recursive=True) + glob.glob(data_root_2 + "*.wav", recursive=True)
    
    elements = []
    
    for f in drones:
        data, sr = librosa.load(f, sr=None, mono=False)
        cast = np.array(data, dtype=np.float32)
        # Check if multiple channels. If yes, split it.
        if cast.ndim > 1:
            for i in range(cast.shape[0]):
                elements.append({"audio": {"array": cast[i], "sampling_rate": sr}, "label": CLASSES["drone"], "src": "dalrd"})
        else:
            elements.append({"audio": {"array": cast, "sampling_rate": sr}, "label": CLASSES["drone"], "src": "dalrd"})
    
    return Dataset.from_list(elements).cast_column("audio", Audio(decode=True))


In [ ]:
def gen_uavirbase():
    data_root = f"{dl_dir}UaVirBASE/Microphone_array/"
    drones = glob.glob(data_root + "*.wav", recursive=True)
    
    elements = []
    
    for f in drones:
        data, sr = librosa.load(f, sr=None, mono=False)
        cast = np.array(data, dtype=np.float32)
        # Check if multiple channels. If yes, split it.
        if cast.ndim > 1:
            for i in range(cast.shape[0]):
                elements.append({"audio": {"array": cast[i], "sampling_rate": sr}, "label": CLASSES["drone"], "src": "uavirbase"})
        else:
            elements.append({"audio": {"array": cast, "sampling_rate": sr}, "label": CLASSES["drone"], "src": "uavirbase"})
    
    return Dataset.from_list(elements).cast_column("audio", Audio()).cast_column("audio", Audio(decode=True))
    

In [ ]:
def gen_yehiellevi(locally_zipped = False):
    data_root = f"{dl_dir}/yehiellevi.dataset-balanced-n-weighted-final/"
    if locally_zipped:
        data_root += "yehiellevi.dataset-balanced-n-weighted-final/"
    
    df = pd.read_csv(data_root + "audio_metadata_shuffled.csv", sep=",")
    
    elements = []
    
    for _, r in df.iterrows():
        path = f"{data_root}/fold{r['fold']}/{r['slice_file_name']}"
        start, end = r["start"], r["end"]
        data, sr = librosa.load(path, sr=None, offset=start, duration=end - start, mono=False)
        cast = np.array(data, dtype=np.float32)
        # Check if multiple channels. If yes, split it.
        if cast.ndim > 1:
            for i in range(cast.shape[0]):
                elements.append({"audio": {"array": cast[i], "sampling_rate": sr}, "label": r["classID"]})
        else:
            elements.append({"audio": {"array": cast, "sampling_rate": sr}, "label": r["classID"], "src": "yehiellevi"})
    
    return Dataset.from_list(elements).cast_column("audio", Audio(decode=True))
    

# NASA's DS is a bit more complicated than the others.

In [ ]:
def load_mat(filepath: str) -> dict:
    """Load a .mat file, trying scipy first (v5), then h5py for v7.3."""
    try:
        mat = sio.loadmat(filepath, squeeze_me=True, struct_as_record=False)
        return mat
    except NotImplementedError:
        # v7.3 HDF5-based .mat files
        try:
            import h5py
        except ImportError:
            print("ERROR: h5py is required for MATLAB v7.3 files. Install with: pip install h5py")
            sys.exit(1)

        def hdf5_to_dict(h5obj):
            result = {}
            for key, val in h5obj.items():
                if isinstance(val, h5py.Dataset):
                    result[key] = val[()]
                elif isinstance(val, h5py.Group):
                    result[key] = hdf5_to_dict(val)
            return result

        import h5py
        with h5py.File(filepath, "r") as f:
            return hdf5_to_dict(f)


In [ ]:
def get_field(struct, field: str):
    """Retrieve a field from either a scipy mat_struct or a plain dict."""
    if isinstance(struct, dict):
        return struct[field]
    return getattr(struct, field)


In [ ]:
def extract_acoustic_data(mat: dict, mic_channel: int):
    """
    Extract acoustic pressure and UTC time from the 'acoustics' structure.

    Returns
    -------
    utc_time : np.ndarray  (n,)   seconds past midnight, UTC
    pressures : np.ndarray (n, num_mics)  incident pressures in Pascals
    selected_channel_pressure : np.ndarray (n,)  pressure for chosen mic
    sample_rate : float  Hz
    """
    acoustics = mat["acoustics"]

    utc_time   = np.asarray(get_field(acoustics, "utc_time"), dtype=float).ravel()
    pressures  = np.asarray(get_field(acoustics, "incident_pascals"), dtype=float)

    # Ensure 2-D: (n_samples, n_mics)
    if pressures.ndim == 1:
        pressures = pressures[:, np.newaxis]

    n_mics = pressures.shape[1]
    if mic_channel >= n_mics:
        raise ValueError(
            f"Requested channel {mic_channel} but data only has {n_mics} mic(s) "
            f"(0-based indexing)."
        )

    selected_pressure = pressures[:, mic_channel]

    # Sampling rate from mean time step
    dt          = np.mean(np.diff(utc_time))
    sample_rate = 1.0 / dt

    return utc_time, pressures, selected_pressure, sample_rate


In [ ]:
def extract_vehicle_data(mat: dict):
    """Extract RTK and GPS vehicle position data."""
    vd = mat.get("vehicle_data")
    if vd is None:
        return None

    result = {}
    for field in ("rtk_utc_time", "rtk_ned_meters", "rtk_status",
                  "gps_utc_time", "gps_ned_meters"):
        try:
            result[field] = np.asarray(get_field(vd, field), dtype=float)
        except (AttributeError, KeyError):
            result[field] = None

    return result


In [ ]:
def extract_met_data(mat: dict):
    """Extract meteorological data if available."""
    met = mat.get("met_data")
    if met is None:
        return None

    result = {}
    for field in ("temperature_celsius", "windspeed_knots", "wind_direction", "utc_time"):
        try:
            result[field] = np.asarray(get_field(met, field), dtype=float)
        except (AttributeError, KeyError):
            result[field] = None

    return result


In [ ]:
def print_summary(utc_time, pressures, selected_pressure, sample_rate,
                  mic_channel, vehicle_data, met_data):
    """Print a human-readable summary to the console."""
    print("=" * 60)
    print("  NASA Small UAV Acoustic Data Extraction")
    print("=" * 60)

    duration = utc_time[-1] - utc_time[0]
    print(f"\n[ACOUSTICS]")
    print(f"  Sample rate      : {sample_rate:.2f} Hz")
    print(f"  Num samples      : {len(utc_time)}")
    print(f"  Duration         : {duration:.2f} s  ({duration/60:.2f} min)")
    print(f"  Num mic channels : {pressures.shape[1]}")
    print(f"  Selected channel : {mic_channel} (0-based)")
    print(f"  Pressure range   : [{selected_pressure.min():.6f}, {selected_pressure.max():.6f}] Pa")
    print(f"  Peak SPL (A-weighted ref 20µPa): "
          f"{20 * np.log10(np.sqrt(np.mean(selected_pressure**2)) / 20e-6):.1f} dB")

    if vehicle_data:
        print(f"\n[VEHICLE DATA]")
        rtk_t = vehicle_data.get("rtk_utc_time")
        gps_t = vehicle_data.get("gps_utc_time")
        ned_rtk = vehicle_data.get("rtk_ned_meters")
        ned_gps = vehicle_data.get("gps_ned_meters")
        if rtk_t is not None:
            print(f"  RTK samples      : {len(rtk_t.ravel())}")
        if gps_t is not None:
            print(f"  GPS samples      : {len(gps_t.ravel())}")
        if ned_rtk is not None and ned_rtk.ndim == 2:
            r = np.sqrt(np.sum(ned_rtk**2, axis=1))
            print(f"  RTK max distance : {r.max():.2f} m from origin")
        if ned_gps is not None and ned_gps.ndim == 2:
            r = np.sqrt(np.sum(ned_gps**2, axis=1))
            print(f"  GPS max distance : {r.max():.2f} m from origin")

    if met_data:
        print(f"\n[METEOROLOGICAL DATA]")
        temp = met_data.get("temperature_celsius")
        wind = met_data.get("windspeed_knots")
        if temp is not None:
            print(f"  Avg temperature  : {np.nanmean(temp):.1f} °C")
        if wind is not None:
            print(f"  Avg wind speed   : {np.nanmean(wind):.1f} kn")
    else:
        print("\n[METEOROLOGICAL DATA] Not available")

    print()


In [ ]:
def extract_nasa_data(mat_file, vehicle: str=None, channel: int=None):
    # Resolve mic channel
    if channel is not None:
        mic_channel = channel
    elif vehicle is not None:
        mic_channel = VEHICLE_MIC_CHANNEL[vehicle]
    else:
        mic_channel = 0
        print(f"No --vehicle or --channel specified; defaulting to channel 0.")

    # Load
    print(f"Loading: {mat_file}")
    mat = load_mat(mat_file)

    # Extract
    utc_time, pressures, selected_pressure, sample_rate = extract_acoustic_data(
        mat, mic_channel
    )

    return pressures[:, mic_channel], int(sample_rate)


In [ ]:
def gen_nasa():
    data_root = f"{dl_dir}/nasa.small_uav_flyover_acoustics/data/"
    files = glob.glob(data_root + "*.mat")

    drones = []

    for f in files:
        basename = os.path.basename(f)
        if basename.startswith("cub_") or basename.startswith("hex_"):
            channel = 0
        else:
            channel = 2
        data, sr = extract_nasa_data(f, channel=channel)
        cast = np.array(data, dtype=np.float32)
        # Check if multiple channels. If yes, split it.
        if cast.ndim > 1:
            for i in range(cast.shape[0]):
                drones.append({"array": cast[i], "sampling_rate": sr})
        else:
            drones.append({"array": cast, "sampling_rate": sr})

    elements = [{"audio": f, "label": CLASSES["drone"], "src": "nasa"} for f in drones]
    return Dataset.from_list(elements).cast_column("audio", Audio(decode=True))


# Load local DSs

# Save what we've got so far.

In [ ]:
_listed = [gen_aira_uas(), gen_ddd(), gen_dalrd(), gen_uavirbase(), gen_yehiellevi(True), gen_nasa()]


In [ ]:
unified_locally = concatenate_datasets(_listed)


In [ ]:
gc.collect()

In [ ]:
unified_locally.save_to_disk(str(local_tmp_dir) + "/local_gathering")


In [ ]:
gc.collect()


In [ ]:
print(unified_locally.features)


# Uniformise the drone DSs

In [ ]:
def ds_split_channels(ds):
    elements = []

    for i in range(len(ds)):
        elem = ds[i]
        sr = elem["audio"]["sampling_rate"]
        audios = np.array(elem["audio"]["array"]).T

        elements += [{"audio": {"sampling_rate": sr, "array": chan}} for chan in audios]

    ds = Dataset.from_list(elements).cast_column("audio", Audio(decode=True))
    return ds.add_column("label", [CLASSES["drone"]] * len(ds))


In [ ]:
def processs_trains(src, start=0):
    _elems = [(k, ds) for k, ds in hf_drone[0][0].items()]
    count = len(_elems)

    for i in range(start, count):
        k, ds = _elems[i]
        print(f"{i}/{count}")
        v = ds_split_channels(ds)
        v.save_to_disk(str(local_tmp_dir) + "/" + k)


In [ ]:
def process_single_train(args):
    i, k, ds, count, local_tmp_dir = args
    print(f"{i}/{count}")
    v = ds_split_channels(ds)
    v.save_to_disk(str(local_tmp_dir) + "/" + k)
    return i, k

def processs_trains(src, start=0, num_threads=4):
    _elems = [(k, ds) for k, ds in src.items()]
    count = len(_elems)

    tasks = [
        (i, k, ds, count, local_tmp_dir)
        for i, (k, ds) in enumerate(_elems)
        if i >= start
    ]

    with ThreadPoolExecutor(max_workers=num_threads) as executor:
        futures = {executor.submit(process_single_train, task): task for task in tasks}
        for future in as_completed(futures):
            try:
                i, k = future.result()
            except Exception as e:
                task = futures[future]
                print(f"Error processing {task[1]}: {e}")


In [ ]:
print(len([(k, ds) for k, ds in hf_drone[0][0].items()]))


In [ ]:
print(hf_drone[0][0])


In [ ]:
processs_trains(hf_drone[0][0], num_threads=8)


In [ ]:
gc.collect()


In [ ]:
def load_splitted_drones_ds(args):
    i, k, count, local_tmp_dir = args
    print(f"{i}/{count}")
    return datasets.load_from_disk(str(local_tmp_dir) + "/" + k)

def load_splitted_drones(src, start=0, num_threads=6):
    _elems = [(k, ds) for k, ds in hf_drone[0][0].items() if k.startswith("train_")]
    count = len(_elems)

    tasks = [
        (i, k, count, local_tmp_dir)
        for i, (k, ds) in enumerate(_elems)
        if i >= start
    ]

    results = []
    with ThreadPoolExecutor(max_workers=num_threads) as executor:
        futures = {executor.submit(load_splitted_drones_ds, task): task for task in tasks}
        for future in as_completed(futures):
            try:
                results.append(future.result())
            except Exception as e:
                task = futures[future]
                print(f"Error processing {task[1]}: {e}")

    return concatenate_datasets(results)


In [ ]:
unified_hf_drones = load_splitted_drones(hf_drone[0][0])


In [ ]:
unified_hf_drones.save_to_disk(str(local_tmp_dir) + "/foreign_drone_gathering")


In [ ]:
gc.collect()


# Uniformise the other DSs

In [ ]:
def gen_others():
    hf_other_simplified = []

    for opt in hf_other:
        if str(type(opt)) == "<class 'datasets.Dataset'>":
            hf_other_simplified.append(opt)
        elif str(type(opt)) == "<class 'datasets.dataset_dict.DatasetDict'>":
            for k, ds in opt.items():
                cols = list(ds.features.keys())
                cols.remove("audio")
    
                hf_other_simplified.append(ds.remove_columns(cols).add_column("label", [CLASSES["other"]] * len(ds)))

    return concatenate_datasets(hf_other_simplified)


In [ ]:
unified_hf_others = gen_others()


In [ ]:
gc.collect()


In [ ]:
unified_hf_others.save_to_disk(str(local_tmp_dir) + "/foreign_other_gathering")


In [ ]:
gc.collect()


# Uniformise the mixed DSs

In [ ]:
def simplify_audio(elem):
    return {"label": elem["label"], "audio": {"sampling_rate": elem["sampling_rate"], "array": np.array(elem["audio"])}}

def simplify_audio(batch):
    return {
        "label": batch["label"],
        "audio": [
            {"sampling_rate": sr, "array": np.array(audio)}
            for sr, audio in zip(batch["sampling_rate"], batch["audio"])
        ]
    }


In [ ]:
def gen_mixed():
    hf_mixed_simplified = []

    for opt in hf_mixed:
        if str(type(opt)) == "<class 'datasets.Dataset'>":
            hf_mixed_simplified.append(opt)
        elif str(type(opt)) == "<class 'datasets.dataset_dict.DatasetDict'>":
            for k, ds in opt.items():
                hf_mixed_simplified.append(ds)

    hf_mixed_pretty = []
    for ds in hf_mixed_simplified:
        if "sampling_rate" in ds.features:
            # Complicated matters
            hf_mixed_pretty.append(ds.map(simplify_audio, batched=True).remove_columns(["sampling_rate"]).cast_column("audio", Audio(decode=True)).cast_column("label", Value("int64")))
        else:
            hf_mixed_pretty.append(ds.cast_column("label", Value("int64")))

    return concatenate_datasets(hf_mixed_pretty)
    

In [ ]:
unified_hf_mixed = gen_mixed()


In [ ]:
gc.collect()

In [ ]:
unified_hf_mixed.save_to_disk(str(local_tmp_dir) + "/foreign_mixed_gathering")


In [ ]:
gc.collect()


# Now, we can properly set everything up for the final stage :)

In [ ]:
stages = ["foreign_mixed_gathering", "foreign_drone_gathering", "foreign_other_gathering", "local_gathering"]


In [ ]:
stages_ds = [datasets.load_from_disk(str(local_tmp_dir) + "/" + stg) for stg in stages]


In [ ]:
print([ds.features["label"] for ds in stages_ds])


In [ ]:
print([ds.features["label"] for ds in stages_ds])


# Just ensure everythin is mono ;)

In [ ]:
def is_valid_audio(example):
    """Filter out problematic audio before processing"""
    try:
        # Quick check if audio exists and has reasonable length
        _array = np.array(example["audio"]["array"])
        return len(_array) > 0 if isinstance(example["audio"], dict) else True
    except:
        return False


In [ ]:
def split_channels(batch):
    new_audios = []
    new_labels = []

    for audio, label in zip(batch["audio"], batch["label"]):
        array = np.array(audio["array"])
        sr = audio["sampling_rate"]

        # Skip anything that is None or empty
        if array is None or array.size == 0:
            continue

        # Handle mono and multi-channel
        if array.ndim == 1 or array.shape[0] == 1:
            new_audios.append(audio)
            new_labels.append(label)
        else:
            for ch in range(array.shape[0]):
                new_audios.append({
                    "array": array[ch, :],
                    "sampling_rate": sr,
                })
                new_labels.append(label)

    return {"audio": new_audios, "label": new_labels}


In [ ]:
label_feature = ClassLabel(names=["other", "drone"])


In [ ]:
i = 3

In [ ]:
for j in range(i, len(stages_ds)):
    i = j
    print("Processing DS", j)

    ds = stages_ds[j].filter(is_valid_audio, num_proc=4).map(split_channels, batched=True, num_proc=4, batch_size=8)
    ds.save_to_disk(str(local_tmp_dir) + "/" + stages[j] + ".mono")


In [ ]:
gc.collect()


# Now we can save it all as one DS.

In [ ]:
result_ds = [datasets.load_from_disk(str(local_tmp_dir) + "/" + stg + ".mono") for stg in stages]


In [ ]:
output_ds = concatenate_datasets(stages_ds)


In [ ]:
num_shards = 325


In [ ]:
for i in range(num_shards):
    print(f"{i}/{num_shards}")
    shard = output_ds.shard(index=i, num_shards=num_shards, contiguous=True)
    shard.to_parquet(f"{LOCAL_DIR}/output/shard_{i:05d}.parquet")


# Try to load it, too see quickly if everything's good.

In [ ]:
output_ds = datasets.load_dataset("parquet", data_dir=f"{LOCAL_DIR}/output/")


In [ ]:
gc.collect()


# We're never sure enough, we'll check that the final DS only contains mono.

In [ ]:
invalids = []
not_mono = []


In [ ]:
def is_mono_audio(audio_dict):
    return audio_dict["array"].ndim == 1  # 1D = mono, 2D = multi-channel


In [ ]:
# Full check
all_mono = True
count = len(output_ds["train"])
j = 0
for example in output_ds["train"]:
    try:
        if not is_mono_audio(example["audio"]):
            all_mono = False
            not_mono.append(j)
    except:
        invalids.append(j)

    j += 1

print(f"All audio is mono: {all_mono}")


In [ ]:
print(invalids)
print(not_mono)


In [ ]:
valid_mask = lambda example, idx: idx not in invalids


In [ ]:
clean_ds = output_ds["train"].filter(valid_mask, with_indices=True).remove_columns(["src"])


In [ ]:
for i in range(num_shards):
    print(f"{i}/{num_shards}")
    shard = clean_ds.shard(index=i, num_shards=num_shards, contiguous=True)
    shard.to_parquet(f"{LOCAL_DIR}/output_cleaned/shard_{i:05d}.parquet")


# Clear the remote repo if required.

In [ ]:
token = ""


In [ ]:
api = HfApi(token=token)


In [ ]:
def clear_repo_files(api: HfApi, repo_id: str, repo_type: str):
    existing_files = set(list_repo_files(repo_id=repo_id, repo_type=repo_type))
    count = len(existing_files)
    i = 0
    for path in existing_files:
        print(f"{i}/{count} {path}")
        api.delete_file(path, repo_id=repo_id, repo_type=repo_type)
        i += 1


In [ ]:
clear_repo_files(api, REPO_ID, REPO_TYPE)


# If you don't have enough API requests, just remove it.

In [ ]:
from huggingface_hub import delete_repo


In [ ]:
delete_repo(token=token, repo_id=REPO_ID, repo_type=REPO_TYPE)


In [ ]:
gc.collect()


# Now we can perform the upload.

In [ ]:
from huggingface_hub import create_repo


In [ ]:
token = ""


In [ ]:
api = HfApi(token=token)


In [ ]:
def upload_directory(api: HfApi, token: str, local_dir: str, repo_id: str, repo_type: str = "dataset"):
    """
    Uploads all files from `local_dir` to the Hugging Face Hub repository.
    Automatically skips already uploaded files (resumable).
    """

    create_repo(repo_id, repo_type=repo_type, token=token, exist_ok=True)

    local_dir = Path(local_dir)
    if not local_dir.exists():
        raise ValueError(f"Local directory does not exist: {local_dir}")

    print(f"📂 Scanning directory: {local_dir}")
    files_to_upload = [p for p in local_dir.rglob("*") if p.is_file()]
    print(f"Found {len(files_to_upload)} files to check.")

    # 🧠 Get list of already uploaded files
    print(f"🔍 Fetching existing files in repo: {repo_id}")
    existing_files = set(list_repo_files(repo_id=repo_id, repo_type=repo_type))
    print(f"Repo already has {len(existing_files)} files.")

    uploaded_count = 0
    skipped_count = 0

    for fpath in files_to_upload:
        # Normalize path in repo (relative to base directory)
        path_in_repo = str(fpath.relative_to(local_dir)).replace("\\", "/")

        if path_in_repo in existing_files:
            print(f"⏩ Skipping already uploaded: {path_in_repo}")
            skipped_count += 1
            continue

        try:
            print(f"⬆️ Uploading: {path_in_repo} ...")
            api.upload_file(
                path_or_fileobj=str(fpath),
                path_in_repo=path_in_repo,
                repo_id=repo_id,
                repo_type=repo_type,
            )
            uploaded_count += 1
            print(f"✅ Uploaded: {path_in_repo}")
        except Exception as e:
            print(f"❌ Error uploading {fpath}: {e}")
            print("Stopping — rerun this script to resume.")
            break

    print("\n✅ Upload complete.")
    print(f"Uploaded: {uploaded_count}, Skipped: {skipped_count}")


In [ ]:
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from huggingface_hub import HfApi, create_repo, list_repo_files


def upload_directory_parrallel(api: HfApi, token: str, local_dir: str, repo_id: str, repo_type: str = "dataset", max_workers: int = 4):
    """
    Uploads all files from `local_dir` to the Hugging Face Hub repository.
    Automatically skips already uploaded files (resumable).
    Uploads up to 4 files in parallel.
    """

    create_repo(repo_id, repo_type=repo_type, token=token, exist_ok=True)

    local_dir = Path(local_dir)
    if not local_dir.exists():
        raise ValueError(f"Local directory does not exist: {local_dir}")

    print(f"📂 Scanning directory: {local_dir}")
    files_to_upload = [p for p in local_dir.rglob("*") if p.is_file()]
    print(f"Found {len(files_to_upload)} files to check.")

    # Get list of already uploaded files
    print(f"🔍 Fetching existing files in repo: {repo_id}")
    existing_files = set(list_repo_files(repo_id=repo_id, repo_type=repo_type))
    print(f"Repo already has {len(existing_files)} files.")

    skipped_count = 0
    upload_tasks = []

    for fpath in files_to_upload:
        path_in_repo = str(fpath.relative_to(local_dir)).replace("\\", "/")

        if path_in_repo in existing_files:
            print(f"⏩ Skipping already uploaded: {path_in_repo}")
            skipped_count += 1
            continue

        upload_tasks.append((fpath, path_in_repo))

    print(f"🚀 Uploading {len(upload_tasks)} files with {max_workers} parallel workers...\n")

    def upload_one(task):
        fpath, path_in_repo = task
        try:
            print(f"⬆️ Uploading: {path_in_repo}")
            api.upload_file(
                path_or_fileobj=str(fpath),
                path_in_repo=path_in_repo,
                repo_id=repo_id,
                repo_type=repo_type,
            )
            return ("ok", path_in_repo)
        except Exception as e:
            return ("error", path_in_repo, str(e))

    uploaded_count = 0

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(upload_one, task) for task in upload_tasks]

        for future in as_completed(futures):
            result = future.result()

            if result[0] == "ok":
                uploaded_count += 1
                print(f"✅ Uploaded: {result[1]}")
            else:
                print(f"❌ Error uploading {result[1]}: {result[2]}")
                print("Stopping — rerun this script to resume.")
                break

    print("\n✅ Upload complete.")
    print(f"Uploaded: {uploaded_count}, Skipped: {skipped_count}")


In [ ]:
upload_directory(api, token, str(LOCAL_DIR) + "/output_cleaned", REPO_ID, REPO_TYPE)
